# HighRes MicroSpin

The HighRes Biosolutions **MicroSpin** is an automation-friendly microplate centrifuge with two
swing-out buckets, one SBS plate in each. A robot puts a plate straight into the bucket the
MicroSpin presents at its door; there is no separate loader.

| | |
|---|---|
| Buckets | 2, one SBS plate each |
| Maximum | 3000 x g |
| Interface | Ethernet, TCP port 1000 |
| Driver | `pylabrobot.high_res.microspin.MicroSpin` |

```{warning}
This driver has been run on one MicroSpin (firmware MS-1.3.3), with empty buckets at 500 x g
only. If it works on yours, or under other loads, please say so on
[discuss.pylabrobot.org](https://discuss.pylabrobot.org) or in a pull request.
```

```{device-card} highres-microspin
```

## How it talks

The MicroSpin answers a text protocol on TCP port 1000. The driver holds two connections to it:
one for the commands that wait their turn on the machine (`home`, presenting a bucket, `spin`,
`status`), and one for those answered at once, so that an abort reaches the device while a spin
runs.

The device's body, its rotor and its two buckets are resources on a resource server. After each
move the driver reads where the rotor and door are and records it there, so the model says which
bucket is at the door.

## Physical setup

Connect the MicroSpin to the host with an Ethernet cable. Its factory address is
`192.168.127.60`, and it scrolls its current address across its display at power-up. Before a
spin, remove the shipping tie-wraps, seat both buckets on the rotor pins, and balance them.

## Connect

`setup()` opens both connections, reads the device's version, and makes the body, rotor and
buckets on the resource server, or finds them there by name. Pass `simulation=True` to try this
page without a device.

In [ ]:
from pylabrobot.high_res.microspin import MicroSpin

cf = MicroSpin(name="microspin", host="192.168.127.60")
await cf.setup()

## Home

After power-on the MicroSpin refuses to move until it is homed.

In [ ]:
await cf.home()
await cf.request_status()

## Present a bucket

`present_bucket` opens the door and turns the rotor so that bucket is at it. The model then says
which bucket is at the door.

In [ ]:
await cf.present_bucket(1)
await cf.get_bucket_at_door()

Put a plate in the presented bucket, by robot or by hand, then present the other bucket and put
the balancing plate in it.

In [ ]:
await cf.present_bucket(2)

## Spin

`spin` closes the door, spins at `g` x g for `duration` seconds at speed, and returns once the
rotor has stopped. A spin with a plate in one bucket and none in the other is refused before
anything is sent.

In [ ]:
await cf.spin(g=500, duration=60)

## Abort and reset

`abort` stops a spin; the rotor then coasts to a stop, about 110 s from 500 x g. Until it has
stopped and the abort latch is released, the driver sends nothing that waits its turn on the
machine: on a device, a command sent during the coast left it answering nothing until it was
power-cycled. `reset` aborts, waits for the rotor, releases the latch, and reads the status.

In [ ]:
await cf.reset()

## Serving it

`python -m pylabrobot.high_res.microspin.server --device 192.168.127.60 --resource-server URL`
serves the MicroSpin where it is plugged in, with its resources on a resource server that other
devices can share (`python -m pylabrobot.resources.server` runs one). Anywhere else, the same
class drives it, with the same methods. A spin is streamed, and asked for again by its request id
if the network drops, so it is never lost or run twice.

```python
cf = MicroSpin(
  microspin_server="http://lab-cf:8082", resource_server="http://lab-resources:8080"
)
await cf.setup()
```

## Stop

`stop()` closes the connections. The resources stay on the resource server.

In [ ]:
await cf.stop()